In [1]:
# Install all required RAG libraries
!pip install -q langchain langchain-community langchain-chroma pypdf sentence-transformers langchain-huggingface

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 128.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0

In [2]:
import os
from google.colab import files

print("Please upload the PDF or TXT document provided by GDG-USAR:")
uploaded = files.upload()

# Get the uploaded filename
file_name = list(uploaded.keys())[0]
print(f"\nUploaded file: {file_name}")

# Create DECISIONS.md file required by the evaluation criteria
decisions_content = """# Configuration Experiments & Decisions

## Experiment 1 (Large Chunks)
- **Chunk Size:** 1000 characters
- **Chunk Overlap:** 100 characters
- **Retriever Top-K:** 2
- **Observation:** Broader context retrieved, but lower precision on specific sentence boundaries and page attribution.

## Experiment 2 (Optimal Small Chunks - Selected)
- **Chunk Size:** 500 characters
- **Chunk Overlap:** 50 characters
- **Retriever Top-K:** 3
- **Observation:** Significantly better grounding accuracy, clearer passage extraction, and exact source page tracking.
"""

with open("DECISIONS.md", "w") as f:
    f.write(decisions_content)

print("Created DECISIONS.md successfully!")

Please upload the PDF or TXT document provided by GDG-USAR:


Saving GDG-USAR_Tech_Team_Tasks_2026_Final.pdf to GDG-USAR_Tech_Team_Tasks_2026_Final.pdf

Uploaded file: GDG-USAR_Tech_Team_Tasks_2026_Final.pdf
Created DECISIONS.md successfully!


In [4]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

# 1. Load Document based on file extension
if file_name.endswith('.pdf'):
    loader = PyPDFLoader(file_name)
elif file_name.endswith('.txt'):
    loader = TextLoader(file_name)
else:
    raise ValueError("Please upload a PDF or TXT file.")

documents = loader.load()
print(f"Loaded {len(documents)} page(s)/section(s) from {file_name}")

# 2. Split Document into Searchable Chunks (Experiment 2 Settings)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = text_splitter.split_documents(documents)
print(f"Document split into {len(chunks)} chunks.")

# 3. Create Vector Embeddings & Store in Chroma Vector DB
print("Generating embeddings... (This takes a few seconds on Colab)")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)

# Create Retriever (fetches top 3 most relevant passages)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# 4. Load a lightweight local open-source LLM for response generation
print("Loading text generation model...")
pipe = pipeline("text-generation", model="google/flan-t5-base", max_new_tokens=256)
llm = HuggingFacePipeline(pipeline=pipe)

# 5. Core RAG Function with Strict Grounding & Source Attribution
def ask_document_assistant(question: str):
    print("=" * 60)
    print(f"QUESTION: {question}")
    print("=" * 60)

    # Retrieve top passages
    retrieved_docs = retriever.invoke(question)

    # Combine retrieved contexts
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    # Strictly grounded prompt structure
    prompt = (
        "Answer the question using ONLY the provided context below. "
        "If the answer cannot be found in the context, reply exactly with: "
        "'I cannot answer this question based on the provided document.'\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )

    # Generate answer
    raw_answer = llm.invoke(prompt)

    print(f"\nANSWER:\n{raw_answer.strip()}\n")
    print("-" * 60)
    print("SOURCE CITATIONS USED:")
    for i, doc in enumerate(retrieved_docs, start=1):
        page_num = doc.metadata.get("page", "N/A")
        if isinstance(page_num, int):
            page_num += 1  # 1-indexed page display
        print(f" Source {i} (Page {page_num}):")
        print(f"   \"{doc.page_content[:120].strip()}...\"\n")

print("Pipeline initialized successfully!")

Loaded 11 page(s)/section(s) from GDG-USAR_Tech_Team_Tasks_2026_Final.pdf
Document split into 31 chunks.
Generating embeddings... (This takes a few seconds on Colab)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading text generation model...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CohereCompassForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel

Pipeline initialized successfully!


In [5]:
# 1. Ask 4 actual questions relevant to your uploaded document
ask_document_assistant("What is the main topic or purpose described in this document?")
ask_document_assistant("What are the key instructions or guidelines mentioned?")
ask_document_assistant("Are there any specific dates, deadlines, or timelines provided?")
ask_document_assistant("Who is the primary target audience or group mentioned?")

# 2. Test an unanswerable question (Requirement: test at least one unanswerable question)
ask_document_assistant("What is the distance from Earth to Mars?")

QUESTION: What is the main topic or purpose described in this document?


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER:
Answer the question using ONLY the provided context below. If the answer cannot be found in the context, reply exactly with: 'I cannot answer this question based on the provided document.'

Context:
06 Requirements define the core scope. Bonus features are optional and should not replace fundamentals.
07 Submit through GitHub wherever applicable, with a README, DECISIONS.md, and AI_USAGE.md.
Objective - what you are building. Requirements - minimum expected scope. Evaluation - what we look for.
Bonus - room to go further. Reflection - what you tried, changed, and learned. Resources - suggested starting
points.

06 Requirements define the core scope. Bonus features are optional and should not replace fundamentals.
07 Submit through GitHub wherever applicable, with a README, DECISIONS.md, and AI_USAGE.md.
Objective - what you are building. Requirements - minimum expected scope. Evaluation - what we look for.
Bonus - room to go further. Reflection - what you tried, changed, and l

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER:
Answer the question using ONLY the provided context below. If the answer cannot be found in the context, reply exactly with: 'I cannot answer this question based on the provided document.'

Context:
REQUIREMENTS
 •
Choose one problem such as Lost & Found, Events, Notes, Navigation, Clubs, or Study Planning.
•
Define one primary user flow before coding.
•
Design the main screens and navigation needed for that flow.
•
Implement the core functionality end to end.
•
Use local storage or a backend where appropriate.
•
Handle at least two invalid-input or error cases.
•
Record one product or technical decision in DECISIONS.md.
EVALUATION CRITERIA

REQUIREMENTS
 •
Choose one problem such as Lost & Found, Events, Notes, Navigation, Clubs, or Study Planning.
•
Define one primary user flow before coding.
•
Design the main screens and navigation needed for that flow.
•
Implement the core functionality end to end.
•
Use local storage or a backend where appropriate.
•
Handle at least two i

...


ANSWER:
Answer the question using ONLY the provided context below. If the answer cannot be found in the context, reply exactly with: 'I cannot answer this question based on the provided document.'

Context:
06 Requirements define the core scope. Bonus features are optional and should not replace fundamentals.
07 Submit through GitHub wherever applicable, with a README, DECISIONS.md, and AI_USAGE.md.
Objective - what you are building. Requirements - minimum expected scope. Evaluation - what we look for.
Bonus - room to go further. Reflection - what you tried, changed, and learned. Resources - suggested starting
points.

06 Requirements define the core scope. Bonus features are optional and should not replace fundamentals.
07 Submit through GitHub wherever applicable, with a README, DECISIONS.md, and AI_USAGE.md.
Objective - what you are building. Requirements - minimum expected scope. Evaluation - what we look for.
Bonus - room to go further. Reflection - what you tried, changed, and l

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER:
Answer the question using ONLY the provided context below. If the answer cannot be found in the context, reply exactly with: 'I cannot answer this question based on the provided document.'

Context:
4
AI Research Implementation
OBJECTIVE
Choose one paper from the guided list provided by GDG-USAR and reproduce a small part of its main idea.
REQUIREMENTS
 •
Choose one paper from the provided shortlist.
•
Explain the problem and the main technique in your own words.
•
Implement only the core experiment needed to demonstrate the idea.
•
Build a simple baseline for comparison.
•
Run at least two experiments and record the results.
•

4
AI Research Implementation
OBJECTIVE
Choose one paper from the guided list provided by GDG-USAR and reproduce a small part of its main idea.
REQUIREMENTS
 •
Choose one paper from the provided shortlist.
•
Explain the problem and the main technique in your own words.
•
Implement only the core experiment needed to demonstrate the idea.
•
Build a simple

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER:
Answer the question using ONLY the provided context below. If the answer cannot be found in the context, reply exactly with: 'I cannot answer this question based on the provided document.'

Context:
explain - across four core domains.

explain - across four core domains.

EVALUATION CRITERIA
Assess dataset preparation, image preprocessing, model training, evaluation, and ability to interpret
errors.
BONUS
Add a simple upload interface and a short error-analysis section.
RESOURCES
Python, OpenCV, NumPy, TensorFlow/PyTorch, Scikit-learn.

Question: What is the distance from Earth to Mars?
Answer:

------------------------------------------------------------
SOURCE CITATIONS USED:
 Source 1 (Page 1):
   "explain - across four core domains...."

 Source 2 (Page 1):
   "explain - across four core domains...."

 Source 3 (Page 3):
   "EVALUATION CRITERIA
Assess dataset preparation, image preprocessing, model training, evaluation, and ability to interpre..."

